# Imports

In [24]:
# Core
import math
import random
import numpy as np
import pickle
from tqdm import tqdm

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim

# TorchText (IMPORTANT: do NOT "import torchtext" to avoid extension issues)
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator

# HuggingFace dataset container
from datasets import Dataset, DatasetDict

# Downloading the text file
import requests
from collections import Counter
import os, pickle


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cpu')

In [3]:
SEED = 1234

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# makes results more reproducible (slower on GPU sometimes)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# Task 1: Dataset Acquisition (1 point)
**Dataset:** Tiny Shakespeare  
**Source credit:** Andrej Karpathy (char-rnn)  
**Link:** https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

**Description:**  
This dataset contains the complete works of William Shakespeare in plain text format. It is commonly used for language modeling because it is text-rich and has consistent style and vocabulary.

**Why chosen:**  
- Rich language and structure (dialogue, narrative, punctuation patterns)  
- Small enough to train quickly for a one-week assignment  
- Suitable for interactive text generation demos


In [4]:
# Direct link to Tiny Shakespeare dataset
shakespeare_url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

# Read the data
text = requests.get(shakespeare_url).text

# Split text into smaller units (lines)
data = text.split("\n")

# Convert to list of dictionaries
data = [{"text": row} for row in data if row.strip() != ""]

# Create HuggingFace Dataset object
dataset = Dataset.from_list(data)

dataset

Dataset({
    features: ['text'],
    num_rows: 32777
})

### Train/Validation/Test Split
We remove empty lines, then split the dataset into:
- **Train:** 80%
- **Validation:** 10%
- **Test:** 10%

A fixed random seed is used to make the split reproducible.


In [50]:
from datasets import DatasetDict
dataset = dataset.filter(lambda x: x["text"] is not None and x["text"].strip() != "")

train_test = dataset.train_test_split(test_size=0.2, seed=SEED)
test_valid = train_test["test"].train_test_split(test_size=0.5, seed=SEED)

dataset = DatasetDict({
    "train": train_test["train"],
    "test": test_valid["test"],
    "validation": test_valid["train"]
})

dataset


Filter:   0%|          | 0/32777 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 26221
    })
    test: Dataset({
        features: ['text'],
        num_rows: 3278
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3278
    })
})

# Model Training (2 point)


### Preprocessing (1 point)
Steps:
1. Remove empty lines.
2. Split into train/validation/test (80/10/10).
3. Tokenize each line using TorchText `basic_english`.
4. Append `<eos>` to indicate the end of a sequence.
5. Build a vocabulary from **training only** (avoid data leakage) with `MIN_FREQ=2`.
6. Convert tokens into integer IDs (`ids`), mapping unknown tokens to `<unk>` (id 0).


In [51]:
tokenizer = get_tokenizer("basic_english")

def tokenize_data(example):
    tokens = tokenizer(example["text"])
    return {"tokens": tokens + ["<eos>"]}

tokenized_dataset = dataset.map(tokenize_data, remove_columns=["text"])
tokenized_dataset


Map:   0%|          | 0/26221 [00:00<?, ? examples/s]

Map:   0%|          | 0/3278 [00:00<?, ? examples/s]

Map:   0%|          | 0/3278 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['tokens'],
        num_rows: 26221
    })
    test: Dataset({
        features: ['tokens'],
        num_rows: 3278
    })
    validation: Dataset({
        features: ['tokens'],
        num_rows: 3278
    })
})

In [52]:
tokenized_dataset["train"][10]["tokens"][:30]


['but',
 ',',
 'sirs',
 ',',
 'be',
 'sudden',
 'in',
 'the',
 'execution',
 ',',
 '<eos>']

### Vocabulary Construction
The vocabulary is built **only from the training split** to prevent validation/test data leakage.  
We keep tokens that appear at least `MIN_FREQ=2` times to control vocabulary size.

Special tokens:
- `<unk>`: unknown token (used when a word is not in the vocabulary)
- `<pad>`: padding token (useful if we later batch sequences of different lengths)
- `<eos>`: end-of-sequence marker


In [79]:
MIN_FREQ = 2
counter = Counter(t for s in tokenized_dataset["train"]["tokens"] for t in s)

specials = ["<unk>", "<pad>", "<eos>"]
itos = specials + [t for t,c in counter.items() if c >= MIN_FREQ and t not in specials]
stoi = {t:i for i,t in enumerate(itos)}

vocab = {"stoi": stoi, "itos": itos}

print("Vocab size:", len(vocab["itos"]))
print("First 10 tokens:", vocab["itos"][:10])


Vocab size: 5772
First 10 tokens: ['<unk>', '<pad>', '<eos>', 'say', ',', 'has', 'our', 'general', 'met', 'the']


In [80]:
import os, pickle

os.makedirs("model", exist_ok=True)

with open("model/vocab_lm.pkl", "wb") as f:
    pickle.dump(vocab, f)

print("Saved vocab to model/vocab_lm.pkl")


Saved vocab to model/vocab_lm.pkl


In [81]:
def numericalize(example):
    ids = [vocab["stoi"].get(t, 0) for t in example["tokens"]]
    return {"ids": ids}

numericalized_dataset = tokenized_dataset.map(numericalize)
numericalized_dataset


Map:   0%|          | 0/26221 [00:00<?, ? examples/s]

Map:   0%|          | 0/3278 [00:00<?, ? examples/s]

Map:   0%|          | 0/3278 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ids'],
        num_rows: 26221
    })
    test: Dataset({
        features: ['tokens', 'ids'],
        num_rows: 3278
    })
    validation: Dataset({
        features: ['tokens', 'ids'],
        num_rows: 3278
    })
})

### Numericalization
Each token list is converted into a list of integer IDs using `vocab["stoi"]`.  
Tokens not present in the vocabulary map to `<unk>` (id 0).  
This produces an `ids` column that is used directly for language model training.


In [82]:

def get_data(hf_dataset_split, batch_size):
    data = []
    for ex in hf_dataset_split:
        if ex["ids"]:
            data.extend(ex["ids"])

    data = torch.LongTensor(data)

    num_batches = data.shape[0] // batch_size
    data = data[:num_batches * batch_size]

    # [batch_size, num_steps]
    data = data.view(batch_size, num_batches)
    return data

BATCH_SIZE = 128

train_data = get_data(numericalized_dataset["train"], BATCH_SIZE)
valid_data = get_data(numericalized_dataset["validation"], BATCH_SIZE)
test_data  = get_data(numericalized_dataset["test"], BATCH_SIZE)

train_data.shape, valid_data.shape, test_data.shape


(torch.Size([128, 1746]), torch.Size([128, 215]), torch.Size([128, 216]))

In [83]:
def get_batch(data, seq_len, idx):
    src = data[:, idx:idx+seq_len]
    tgt = data[:, idx+1:idx+seq_len+1]
    return src, tgt


### Model Architecture and Training Process (1 point)

**Architecture:**  
Embedding → LSTM (2 layers) → Linear layer (predict next token)

**Training objective:**  
Given a sequence of tokens, predict the next token at each time step (teacher forcing).

**Loss:**  
CrossEntropyLoss over vocabulary.

**Metric:**  
Perplexity = exp(loss), lower is better.

**Stability techniques:**  
Gradient clipping prevents exploding gradients.


In [84]:

class LSTMLanguageModel(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, num_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.lstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hid_dim,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True
        )
        self.fc = nn.Linear(hid_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src, hidden):
        emb = self.dropout(self.embedding(src))
        out, hidden = self.lstm(emb, hidden)
        out = self.dropout(out)
        pred = self.fc(out)  # [B, S, V]
        return pred, hidden

    def init_hidden(self, batch_size, device, num_layers, hid_dim):
        h = torch.zeros(num_layers, batch_size, hid_dim).to(device)
        c = torch.zeros(num_layers, batch_size, hid_dim).to(device)
        return (h, c)

    def detach_hidden(self, hidden):
        return (hidden[0].detach(), hidden[1].detach())


In [85]:
VOCAB_SIZE = len(vocab["itos"])   
EMB_DIM = 256
HID_DIM = 512
NUM_LAYERS = 2
DROPOUT = 0.3

model = LSTMLanguageModel(VOCAB_SIZE, EMB_DIM, HID_DIM, NUM_LAYERS, DROPOUT).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

model


LSTMLanguageModel(
  (embedding): Embedding(5772, 256)
  (lstm): LSTM(256, 512, num_layers=2, batch_first=True, dropout=0.3)
  (fc): Linear(in_features=512, out_features=5772, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
)

In [86]:

def train(model, data, optimizer, criterion, batch_size, seq_len, clip, device):
    model.train()
    epoch_loss = 0

    num_steps = data.shape[1]
    data = data[:, :num_steps - (num_steps - 1) % seq_len]
    num_steps = data.shape[1]

    hidden = model.init_hidden(batch_size, device, NUM_LAYERS, HID_DIM)

    for idx in tqdm(range(0, num_steps - 1, seq_len), desc="Training", leave=False):
        optimizer.zero_grad()
        hidden = model.detach_hidden(hidden)

        src, tgt = get_batch(data, seq_len, idx)
        src, tgt = src.to(device), tgt.to(device)

        pred, hidden = model(src, hidden)

        pred = pred.reshape(-1, VOCAB_SIZE)
        tgt = tgt.reshape(-1)

        loss = criterion(pred, tgt)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()

        epoch_loss += loss.item() * seq_len

    return epoch_loss / num_steps


In [87]:
import math
@torch.no_grad()
def evaluate(model, data, criterion, batch_size, seq_len, device):
    model.eval()
    epoch_loss = 0

    num_steps = data.shape[1]
    data = data[:, :num_steps - (num_steps - 1) % seq_len]
    num_steps = data.shape[1]

    hidden = model.init_hidden(batch_size, device, NUM_LAYERS, HID_DIM)

    for idx in tqdm(range(0, num_steps - 1, seq_len), desc="Evaluating", leave=False):
        src, tgt = get_batch(data, seq_len, idx)
        src, tgt = src.to(device), tgt.to(device)

        pred, hidden = model(src, hidden)

        pred = pred.reshape(-1, VOCAB_SIZE)
        tgt = tgt.reshape(-1)

        loss = criterion(pred, tgt)
        epoch_loss += loss.item() * seq_len

    avg_loss = epoch_loss / num_steps
    ppl = math.exp(avg_loss)
    return avg_loss, ppl


In [88]:
SEQ_LEN = 35
CLIP = 1.0
EPOCHS = 8

best_valid_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    train_loss = train(model, train_data, optimizer, criterion, BATCH_SIZE, SEQ_LEN, CLIP, device)
    valid_loss, valid_ppl = evaluate(model, valid_data, criterion, BATCH_SIZE, SEQ_LEN, device)

    print(f"Epoch {epoch:02d} | train loss {train_loss:.3f} | valid loss {valid_loss:.3f} | valid ppl {valid_ppl:.2f}")

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), "model/lstm_lm.pt")
        print("Saved best model to model/lstm_lm.pt")


Epoch 01 | train loss 6.110 | valid loss 5.534 | valid ppl 253.22
Saved best model to model/lstm_lm.pt


Epoch 02 | train loss 5.578 | valid loss 5.247 | valid ppl 190.03
Saved best model to model/lstm_lm.pt


Epoch 03 | train loss 5.295 | valid loss 5.040 | valid ppl 154.43
Saved best model to model/lstm_lm.pt


Epoch 04 | train loss 5.130 | valid loss 4.907 | valid ppl 135.30
Saved best model to model/lstm_lm.pt


Epoch 05 | train loss 4.993 | valid loss 4.796 | valid ppl 120.99
Saved best model to model/lstm_lm.pt


Epoch 06 | train loss 4.883 | valid loss 4.716 | valid ppl 111.68
Saved best model to model/lstm_lm.pt


Epoch 07 | train loss 4.800 | valid loss 4.670 | valid ppl 106.73
Saved best model to model/lstm_lm.pt


Epoch 08 | train loss 4.734 | valid loss 4.620 | valid ppl 101.46
Saved best model to model/lstm_lm.pt


In [89]:
model.load_state_dict(torch.load("model/lstm_lm.pt", map_location=device))
test_loss, test_ppl = evaluate(model, test_data, criterion, BATCH_SIZE, SEQ_LEN, device)
print(f"Test loss: {test_loss:.3f} | Test perplexity: {test_ppl:.2f}")


/var/folders/k1/5yp6fwzs6810rxp0qptj32bm0000gq/T/ipykernel_1225/2397194281.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("model/lstm_l

Test loss: 4.615 | Test perplexity: 100.95


# Text Generation - Web Application Development (2 points)

## Text Generation & Web Application (2 points)

### Text Generation
The model generates text by taking a user prompt, converting it into token IDs, and sampling next tokens from the model’s output distribution (controlled by temperature).

### Web App Interface 
The Flask app loads:
- `model/lstm_lm.pt` (trained weights)
- `model/vocab_lm.pkl` (stoi/itos mappings)

Flow:
1. User enters prompt in an input box.
2. Prompt is tokenized (`basic_english`) and converted to IDs using `stoi`.
3. Model generates a continuation.
4. Generated IDs are decoded back into text using `itos`.
5. Output is displayed in the web page.


In [90]:
@torch.no_grad()
def generate(prompt, max_new_tokens, temperature, model, tokenizer, vocab, device, seed=1234):
    torch.manual_seed(seed)
    model.eval()

    tokens = tokenizer(prompt)
    if len(tokens) == 0:
        tokens = ["<unk>"]

    ids = [vocab["stoi"].get(t, 0) for t in tokens]  # 0 = <unk>
    hidden = model.init_hidden(batch_size=1, device=device, num_layers=NUM_LAYERS, hid_dim=HID_DIM)

    # warm-up
    x = torch.tensor(ids, dtype=torch.long).unsqueeze(0).to(device)  # [1, T]
    _, hidden = model(x, hidden)

    last_id = ids[-1]
    out_ids = ids[:]

    for _ in range(max_new_tokens):
        x = torch.tensor([[last_id]], dtype=torch.long).to(device)   # [1,1]
        pred, hidden = model(x, hidden)                              # [1,1,V]

        logits = pred[0, -1] / max(temperature, 1e-8)
        probs = torch.softmax(logits, dim=-1)

        next_id = torch.multinomial(probs, 1).item()
        out_ids.append(next_id)
        last_id = next_id

    words = [vocab["itos"][i] for i in out_ids if vocab["itos"][i] != "<eos>"]
    words = ["[UNK]" if w == "<unk>" else w for w in words]
    return " ".join(words)




In [91]:
print(generate(
    prompt="to be or not to be",
    max_new_tokens=40,
    temperature=0.9,
    model=model,
    tokenizer=tokenizer,
    vocab=vocab,
    device=device
))


to be or not to be ? leave ' ll have me , away ! the heart [UNK] the man are not act in the man , to do him at [UNK] ' d eyes ? away , [UNK] , bid it
